# The small run, stage by stage

d=3, 12 rounds, 3 sliding windows, p=0.001, seed 8: one detector fires in
round 11 and the observable really flips. Each cell prints what one stage
receives and what it sends on, with timestamps. Hand formulas: README
section 1.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "guide" / "walkthrough"))
sys.path.insert(0, str(repo_root))

from experiments.baseline.baseline_closed_loop import build_run, load_config

CONFIG_PATH = repo_root / "experiments/validation/analytic_oracle_d3_r12.yaml"
print(CONFIG_PATH.read_text())

# Deterministic analytical-oracle case.
#
# Purpose:
#   Verify the exact baseline event ordering and timing, not performance or LER.
#   This case intentionally has:
#     - zero physical noise,
#     - one fixed decoder latency,
#     - one decoder unit,
#     - unbounded link bandwidth,
#     - only 12 QEC rounds, which produce exactly 3 sliding windows.
#
# Expected analytical results are provided beside this file.

code_task: surface_code:rotated_memory_z
distance: 3
rounds_per_shot: 12

windowing:
  scheme: sliding
  commit_rounds: 3
  buffer_rounds: 3

sweep:
  # p = 0 exactly cannot run: a noiseless circuit has an empty detector error
  # model and no matching graph. 1.0e-9 samples zero defects on every shot and
  # every timing tick is identical to p = 0 (preset stage costs, payload sizes
  # independent of p).
  - physical_error_probability: [1.0e-9]
    round_period_us: [1.0]
    algorithm_latency_us: [0.028]
    shots: 1

controller:
  t_binary_availability_us: 0.0
  t_pack

## Run the shot

In [2]:
config = load_config(CONFIG_PATH)
spec, decoder_engine = build_run(config, physical_error_probability=0.001,
                                 round_period_us=1.0, algorithm_latency_us=0.028, seed=8)
completed = spec.build()
print(completed.result.terminal_status)

complete


## The records the run left behind (every cell below only reads these)

In [3]:
from decsim.config import TICKS_PER_US

transfers = completed.result.link_traffic["transfers"]
windows = {window_id: window
           for (_, window_id), window in sorted(completed.window_manager.windows.items())}
frame_records = {record.window_key[1]: record
                 for record in completed.pauli_frame.snapshot().records}
operation = spec.ops[0]
qpu = completed.qpu.model


def us(ticks):
    return round(ticks / TICKS_PER_US, 6)


def transfers_on(path):
    """round -> transfer for qc and c2b, window -> transfer for the rest."""
    selected = {}
    for transfer in transfers:
        if transfer["path"] != path:
            continue
        if path in ("qc", "c2b"):
            selected[transfer["attribution"]["round_lo"]] = transfer
        else:
            selected[transfer["attribution"]["window_id"]] = transfer
    return selected


qc = transfers_on("qc")
c2b = transfers_on("c2b")
cwd = transfers_on("cwd")
dd = transfers_on("dd")
wdo = transfers_on("wdo")


def bits_of_round(round_index):
    fragment = qpu.round_payloads(operation, round_index)[0]
    return "".join(str(int(bit)) for bit in fragment.bits)


print("paths that carried traffic:", sorted({t["path"] for t in transfers}))

paths that carried traffic: ['c2b', 'cwd', 'dd', 'qc', 'wdo']


## Stage 1: the QPU

IN: the circuit below, one round per microsecond (the controller sends the
QPU nothing at runtime, there is no cq traffic). OUT: one readout per
round, that round's detector bits.

In [4]:
print(operation.circuit)

QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(1, 3) 8
QUBIT_COORDS(2, 2) 9
QUBIT_COORDS(3, 3) 10
QUBIT_COORDS(4, 2) 11
QUBIT_COORDS(5, 3) 12
QUBIT_COORDS(6, 2) 13
QUBIT_COORDS(0, 4) 14
QUBIT_COORDS(1, 5) 15
QUBIT_COORDS(2, 4) 16
QUBIT_COORDS(3, 5) 17
QUBIT_COORDS(4, 4) 18
QUBIT_COORDS(5, 5) 19
QUBIT_COORDS(4, 6) 25
R 1 3 5 8 10 12 15 17 19
X_ERROR(0.001) 1 3 5 8 10 12 15 17 19
R 2 9 11 13 14 16 18 25
X_ERROR(0.001) 2 9 11 13 14 16 18 25
TICK
DEPOLARIZE1(0.001) 1 3 5 8 10 12 15 17 19
H 2 11 16 25
DEPOLARIZE1(0.001) 2 11 16 25
TICK
CX 2 3 16 17 11 12 15 14 10 9 19 18
DEPOLARIZE2(0.001) 2 3 16 17 11 12 15 14 10 9 19 18
TICK
CX 2 1 16 15 11 10 8 14 3 9 12 18
DEPOLARIZE2(0.001) 2 1 16 15 11 10 8 14 3 9 12 18
TICK
CX 16 10 11 5 25 19 8 9 17 18 12 13
DEPOLARIZE2(0.001) 16 10 11 5 25 19 8 9 17 18 12 13
TICK
CX 16 8 11 3 25 17 1 9 10 18 5 13
DEPOLARIZE2(0.001) 16 8 11 3 25 17 1 9 10 18 5 13
TICK
H 2 11 16 25
DEPOLARIZE1(0.001) 2 11 16 25
TICK
X

In [5]:
for round_index in sorted(qc):
    time_sent = us(qc[round_index]["send_ticks"])
    print("round", round_index, "leaves QPU at", time_sent, "µs  bits:", bits_of_round(round_index))

round 1 leaves QPU at 1.0 µs  bits: 0000
round 2 leaves QPU at 2.0 µs  bits: 00000000
round 3 leaves QPU at 3.0 µs  bits: 00000000
round 4 leaves QPU at 4.0 µs  bits: 00000000
round 5 leaves QPU at 5.0 µs  bits: 00000000
round 6 leaves QPU at 6.0 µs  bits: 00000000
round 7 leaves QPU at 7.0 µs  bits: 00000000
round 8 leaves QPU at 8.0 µs  bits: 00000000
round 9 leaves QPU at 9.0 µs  bits: 00000000
round 10 leaves QPU at 10.0 µs  bits: 00000000
round 11 leaves QPU at 11.0 µs  bits: 00010000
round 12 leaves QPU at 12.0 µs  bits: 000000000000


## Stage 2: the QC link

IN: the readout at its send time. OUT: the same bits at the controller
0.15 µs later.

In [6]:
for round_index in sorted(qc):
    transfer = qc[round_index]
    sent = us(transfer["send_ticks"])
    arrived = us(transfer["delivery_ticks"])
    print("round", round_index, "sent", sent, "µs  at controller", arrived, "µs")

round 1 sent 1.0 µs  at controller 1.15 µs
round 2 sent 2.0 µs  at controller 2.15 µs
round 3 sent 3.0 µs  at controller 3.15 µs
round 4 sent 4.0 µs  at controller 4.15 µs
round 5 sent 5.0 µs  at controller 5.15 µs
round 6 sent 6.0 µs  at controller 6.15 µs
round 7 sent 7.0 µs  at controller 7.15 µs
round 8 sent 8.0 µs  at controller 8.15 µs
round 9 sent 9.0 µs  at controller 9.15 µs
round 10 sent 10.0 µs  at controller 10.15 µs
round 11 sent 11.0 µs  at controller 11.15 µs
round 12 sent 12.0 µs  at controller 12.15 µs


## Stage 3: controller processing (pulses to binary)

t_binary_availability_us is 0.0 in this yaml, so the bits are
binary-available the moment they arrive.

In [7]:
print("t_binary_availability_us =", config["controller"]["t_binary_availability_us"])
arrived = us(qc[11]["delivery_ticks"])
print("round 11 arrived at", arrived, "µs and is binary-available at", arrived, "µs")

t_binary_availability_us = 0.0
round 11 arrived at 11.15 µs and is binary-available at 11.15 µs


## Stage 4: syndrome packing

IN: the round's fragments (this device emits one per round, so nothing
waits). OUT: one packed round, same bits, at the same time (t_pack 0.0).

In [8]:
print("t_pack_us =", config["controller"]["t_pack_us"])
for round_index in sorted(qc):
    fragment = qpu.round_payloads(operation, round_index)[0]
    packed_at = us(c2b[round_index]["send_ticks"])
    print("round", round_index, "fragments in:", fragment.n_fragments,
          " bits:", bits_of_round(round_index), " packed at", packed_at, "µs")

t_pack_us = 0.0
round 1 fragments in: 1  bits: 0000  packed at 1.15 µs
round 2 fragments in: 1  bits: 00000000  packed at 2.15 µs
round 3 fragments in: 1  bits: 00000000  packed at 3.15 µs
round 4 fragments in: 1  bits: 00000000  packed at 4.15 µs
round 5 fragments in: 1  bits: 00000000  packed at 5.15 µs
round 6 fragments in: 1  bits: 00000000  packed at 6.15 µs
round 7 fragments in: 1  bits: 00000000  packed at 7.15 µs
round 8 fragments in: 1  bits: 00000000  packed at 8.15 µs
round 9 fragments in: 1  bits: 00000000  packed at 9.15 µs
round 10 fragments in: 1  bits: 00000000  packed at 10.15 µs
round 11 fragments in: 1  bits: 00010000  packed at 11.15 µs
round 12 fragments in: 1  bits: 000000000000  packed at 12.15 µs


## Stage 5: the C2B link into Buffer 0

IN: the packed round. OUT: the round published in Buffer 0 0.10 µs later.
Every round is published 0.25 µs after it left the QPU: links shift the
phase, never the rate.

In [9]:
for round_index in sorted(c2b):
    left_qpu = us(qc[round_index]["send_ticks"])
    published = us(c2b[round_index]["delivery_ticks"])
    print("round", round_index, "left QPU", left_qpu, "µs  published in Buffer 0", published, "µs")

round 1 left QPU 1.0 µs  published in Buffer 0 1.25 µs
round 2 left QPU 2.0 µs  published in Buffer 0 2.25 µs
round 3 left QPU 3.0 µs  published in Buffer 0 3.25 µs
round 4 left QPU 4.0 µs  published in Buffer 0 4.25 µs
round 5 left QPU 5.0 µs  published in Buffer 0 5.25 µs
round 6 left QPU 6.0 µs  published in Buffer 0 6.25 µs
round 7 left QPU 7.0 µs  published in Buffer 0 7.25 µs
round 8 left QPU 8.0 µs  published in Buffer 0 8.25 µs
round 9 left QPU 9.0 µs  published in Buffer 0 9.25 µs
round 10 left QPU 10.0 µs  published in Buffer 0 10.25 µs
round 11 left QPU 11.0 µs  published in Buffer 0 11.25 µs
round 12 left QPU 12.0 µs  published in Buffer 0 12.25 µs


## Stage 6: the window manager

IN: the published rounds. OUT: one decode job per window, whose data is
its rounds' bits side by side. Window 2's bits contain the fired bit. A
window is ready when its last round is published, queued when the previous
window's DD handoff has arrived, dispatched when the unit is free.

In [10]:
last_round = config["rounds_per_shot"]

window_bits = {}
for window_id, window in windows.items():
    first_round = window.start_round
    final_round = min(window.buffer_hi, last_round)
    rounds = range(first_round, final_round + 1)
    window_bits[window_id] = "".join(bits_of_round(round_index) for round_index in rounds)
    print("window", window_id, "reads rounds", first_round, "to", final_round,
          " commits", window.commit_lo, "to", window.commit_hi)
    print("  bits:", window_bits[window_id])
    print("  ready", us(window.t_data_complete), "µs  queued", us(window.t_queued),
          "µs  dispatch", us(window.t_dispatch), "µs")

window 0 reads rounds 1 to 6  commits 1 to 3
  bits: 00000000000000000000000000000000000000000000
  ready 6.25 µs  queued 6.25 µs  dispatch 6.25 µs
window 1 reads rounds 4 to 9  commits 4 to 6
  bits: 000000000000000000000000000000000000000000000000
  ready 9.25 µs  queued 9.25 µs  dispatch 9.25 µs
window 2 reads rounds 7 to 12  commits 7 to 12
  bits: 0000000000000000000000000000000000010000000000000000
  ready 12.25 µs  queued 12.25 µs  dispatch 12.25 µs


## Stage 7: the CWD link into decoder memory

IN: the dispatched window's bits out of Buffer 0. OUT: the same bits in
the decoder unit's memory 2.0 µs later.

In [11]:
for window_id in windows:
    transfer = cwd[window_id]
    left = us(transfer["send_ticks"])
    arrived = us(transfer["delivery_ticks"])
    print("window", window_id, "bits:", transfer["payload_bits"],
          " left Buffer 0", left, "µs  in decoder memory", arrived, "µs")

window 0 bits: 44  left Buffer 0 6.25 µs  in decoder memory 8.25 µs
window 1 bits: 48  left Buffer 0 9.25 µs  in decoder memory 11.25 µs
window 2 bits: 52  left Buffer 0 12.25 µs  in decoder memory 14.25 µs


## Stage 8: the decoder engine

IN: the window's bits. Fetch reads them out of memory, the algorithm card
charges 0.028 µs, release writes the result out. OUT: the window's
correction. Window 2 saw the fired bit and outputs 1.

In [12]:
for window_id in windows:
    for record in decoder_engine.stage_records_for(operation.id, window_id):
        print("window", window_id, record.stage, "from", us(record.start_ticks),
              "to", us(record.end_ticks), "µs")
    correction = frame_records[window_id].logical_observables
    print("window", window_id, "correction:", correction)
    print()

window 0 fetch from 8.25 to 8.274 µs
window 0 algorithm from 8.274 to 8.302 µs
window 0 release from 8.302 to 8.306 µs
window 0 correction: (0,)

window 1 fetch from 11.25 to 11.274 µs
window 1 algorithm from 11.274 to 11.302 µs
window 1 release from 11.302 to 11.306 µs
window 1 correction: (0,)

window 2 fetch from 14.25 to 14.274 µs
window 2 algorithm from 14.274 to 14.302 µs
window 2 release from 14.302 to 14.306 µs
window 2 correction: (1,)



## Stage 9: the DD handoff

OUT: the decoded window's boundary to the next window's decode, 0.5 µs
after decode done. Its delivery is the dependency arrival shown in stage
6. The last window has no successor.

In [13]:
for window_id in windows:
    if window_id not in dd:
        print("window", window_id, "is last, no handoff")
        continue
    transfer = dd[window_id]
    sent = us(transfer["send_ticks"])
    delivered = us(transfer["delivery_ticks"])
    print("window", window_id, "sent", sent, "µs  delivered to next window", delivered, "µs")

window 0 sent 8.306 µs  delivered to next window 8.806 µs
window 1 sent 11.306 µs  delivered to next window 11.806 µs
window 2 is last, no handoff


## Stage 10: the WDO link

OUT: the correction at the Pauli frame, 1.0 µs after decode done.

In [14]:
for window_id, record in sorted(frame_records.items()):
    transfer = wdo[window_id]
    sent = us(transfer["send_ticks"])
    arrived = us(transfer["delivery_ticks"])
    print("window", window_id, "correction", record.logical_observables,
          " sent", sent, "µs  at frame", arrived, "µs")

window 0 correction (0,)  sent 8.306 µs  at frame 9.306 µs
window 1 correction (0,)  sent 11.306 µs  at frame 12.306 µs
window 2 correction (1,)  sent 14.306 µs  at frame 15.306 µs


## Stage 11: the Pauli frame

IN: the corrections, in window order. OUT: the running frame, the XOR of
everything committed so far. The final frame is the loop's prediction; it
equals the QPU's sampled truth, so the flipped observable was decoded
correctly.

In [15]:
frame = 0
for window_id, record in sorted(frame_records.items()):
    frame = frame ^ record.logical_observables[0]
    print("window", window_id, "correction", record.logical_observables,
          " committed at", us(record.committed_ticks), "µs  frame is now", (frame,))

result = completed.result.operation_results[0]
print()
print("loop prediction:", tuple(result.logical_observables))
print("observable truth:", tuple(result.observable_truth))

window 0 correction (0,)  committed at 9.31 µs  frame is now (0,)
window 1 correction (0,)  committed at 12.31 µs  frame is now (0,)
window 2 correction (1,)  committed at 15.31 µs  frame is now (1,)

loop prediction: (1,)
observable truth: (1,)


## End to end, against the hand arithmetic

The run condensed to one row per window, next to README section 1's
closed-form answer key. The defect changed the data, not one timestamp.

In [16]:
from analytic_small_run import analytic_timeline, print_table
from simulated_small_run import differences, simulated_timeline

simulated_rows = simulated_timeline(config)
print("Simulator:")
print_table(simulated_rows)
print()
print("By hand:")
print_table(analytic_timeline(config))
print()

disagreements = differences(analytic_timeline(config), simulated_rows)
if disagreements:
    print("MISMATCHES:", disagreements)
else:
    print("MATCH: every cell agrees to the tick.")

Simulator:
| window_id | read_lo | read_hi | commit_lo | commit_hi | buffer0_ready_us | queued_us | dispatch_us | decode_done_us | dd_delivery_us | frame_commit_us | buffer0_ready_to_frame_us |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 6 | 1 | 3 | 6.250 | 6.250 | 6.250 | 8.306 | 8.806 | 9.310 | 3.060 |
| 1 | 4 | 9 | 4 | 6 | 9.250 | 9.250 | 9.250 | 11.306 | 11.806 | 12.310 | 3.060 |
| 2 | 7 | 12 | 7 | 12 | 12.250 | 12.250 | 12.250 | 14.306 |  | 15.310 | 3.060 |

By hand:
| window_id | read_lo | read_hi | commit_lo | commit_hi | buffer0_ready_us | queued_us | dispatch_us | decode_done_us | dd_delivery_us | frame_commit_us | buffer0_ready_to_frame_us |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 6 | 1 | 3 | 6.250 | 6.250 | 6.250 | 8.306 | 8.806 | 9.310 | 3.060 |
| 1 | 4 | 9 | 4 | 6 | 9.250 | 9.250 | 9.250 | 11.306 | 11.806 | 12.310 | 3.060 |
| 2 | 7 | 12 | 7 | 12 | 12.250 | 12.250 | 12.250 | 14.306 |  | 15.310 | 3.060 |

MATCH: every cell agrees to the ti